# 04 - Evaluate Trained MLP on S3DIS Features

This notebook runs quantitative evaluation for the trained translation head on pre-extracted S3DIS features.

It expects feature `.npz` files and trained `.pth` checkpoints to already exist on the shared Google Drive. Choose the feature folder for the target area below.

### 1. Setup Repo & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
DRIVE_ROOT = '/content/drive/MyDrive/DL_Project'
EVAL_FEATURES_NAME = 's3dis_area4'
DRIVE_FEATURES_DIR = f'{DRIVE_ROOT}/features/{EVAL_FEATURES_NAME}'
LOCAL_FEATURES_ROOT = '/content/local_features'
LOCAL_FEATURES_DIR = f'{LOCAL_FEATURES_ROOT}/{EVAL_FEATURES_NAME}'
BRANCH = 'dev/eval-demo'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}


In [ ]:
%cd {REPO_DIR}
!git restore configs/train_mlp_s3dis.yaml notebooks/pyproject.toml
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull --no-edit origin {BRANCH}
!git branch --show-current
!git log -1 --oneline

### 2. Wire Drive Paths and Copy Features Locally

We keep `data/` and `checkpoints/` on Drive, but copy the selected feature folder to local runtime storage for faster evaluation.

In [ ]:
%cd {REPO_DIR}
!rm -f data checkpoints features pretrained
!ln -sf {DRIVE_ROOT}/data ./data
!ln -sf {DRIVE_ROOT}/checkpoints ./checkpoints
!mkdir -p {LOCAL_FEATURES_ROOT}
!rm -rf {LOCAL_FEATURES_DIR}
!cp -r {DRIVE_FEATURES_DIR} {LOCAL_FEATURES_ROOT}/
!ln -sf {LOCAL_FEATURES_ROOT} ./features
!echo Selected feature folder: {EVAL_FEATURES_NAME}
!readlink -f ./features/{EVAL_FEATURES_NAME}
!du -sh {LOCAL_FEATURES_DIR}
!ls -lh ./checkpoints/mlp_s3dis | tail

### 3. Setup Environment & Hugging Face Token

In [ ]:
%cd /content/Deep_learning_project/notebooks

from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
with open('.env', 'w') as f:
    f.write(f'HF_TOKEN={HF_TOKEN}\n')

!uv python install 3.10.12
!uv sync --python 3.10.12

### 4. Choose a Checkpoint and Run Evaluation

Set `CHECKPOINT_NAME` to the checkpoint you want to evaluate, for example `epoch_040.pth` or `last_model.pth`.

In [ ]:
CONFIG_PATH = '/content/Deep_learning_project/configs/train_mlp_s3dis.yaml'
EVAL_BATCH_SIZE = 16384
RESULTS_DIR = '/content/drive/MyDrive/DL_Project/results/evaluation'
CHECKPOINT_NAME = 'last_model.pth'
CHECKPOINT_PATH = f'/content/Deep_learning_project/checkpoints/mlp_s3dis/{CHECKPOINT_NAME}'
print('Config:', CONFIG_PATH)
print('Eval batch size:', EVAL_BATCH_SIZE)
print('Results dir:', RESULTS_DIR)
print('Features dir:', f'/content/Deep_learning_project/features/{EVAL_FEATURES_NAME}')
print('Checkpoint:', CHECKPOINT_PATH)

!PYTHONPATH=/content/Deep_learning_project uv run python /content/Deep_learning_project/src/evaluate.py --config {CONFIG_PATH} --features_dir /content/Deep_learning_project/features/{EVAL_FEATURES_NAME} --checkpoint {CHECKPOINT_PATH} --batch_size {EVAL_BATCH_SIZE} --results_dir {RESULTS_DIR}

### 5. Visualize the Saved Evaluation Results

In [ ]:
import json
import math
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

AREA_JSON_PATH = Path(RESULTS_DIR) / 'area_level' / f'{EVAL_FEATURES_NAME}_{Path(CHECKPOINT_NAME).stem}_metrics.json'
ROOM_CSV_PATH = Path(RESULTS_DIR) / 'room_level' / f'{EVAL_FEATURES_NAME}_{Path(CHECKPOINT_NAME).stem}_room_metrics.csv'

print('Area metrics file:', AREA_JSON_PATH)
print('Room metrics file:', ROOM_CSV_PATH)

with AREA_JSON_PATH.open('r', encoding='utf-8') as handle:
    area_metrics = json.load(handle)

class_names = area_metrics['class_names']
class_ious = [float('nan') if value is None else float(value) for value in area_metrics['per_class_iou']]

plt.figure(figsize=(12, 5))
plt.bar(range(len(class_names)), class_ious)
plt.xticks(range(len(class_names)), class_names, rotation=45, ha='right')
plt.ylabel('IoU')
plt.ylim(0.0, 1.0)
plt.title(f'Per-class IoU - {EVAL_FEATURES_NAME} - {Path(CHECKPOINT_NAME).stem}')
plt.tight_layout()
plt.show()

In [ ]:
room_df = pd.read_csv(ROOM_CSV_PATH)
room_df = room_df.sort_values('miou', ascending=False).reset_index(drop=True)

display(room_df[['room_file', 'num_points', 'oa', 'miou']].head(10))

plt.figure(figsize=(14, 5))
plt.bar(room_df['room_file'], room_df['miou'])
plt.xticks(rotation=90)
plt.ylabel('mIoU')
plt.ylim(0.0, 1.0)
plt.title(f'Room-level mIoU - {EVAL_FEATURES_NAME} - {Path(CHECKPOINT_NAME).stem}')
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))
plt.bar(room_df['room_file'], room_df['oa'])
plt.xticks(rotation=90)
plt.ylabel('Overall Accuracy')
plt.ylim(0.0, 1.0)
plt.title(f'Room-level OA - {EVAL_FEATURES_NAME} - {Path(CHECKPOINT_NAME).stem}')
plt.tight_layout()
plt.show()